# Лабораторная работа №4
## Линейные модели, SVM и деревья решений

Методичка: [LAB_TMO_TREES](https://github.com/ugapanyuk/courses_current/wiki/LAB_TMO_TREES).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, f1_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
%matplotlib inline

RANDOM_STATE = 42
wine = load_wine(as_frame=True)
X = wine.data
y = wine.target
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

In [ ]:
models = {
    "LogReg": Pipeline([( "scaler", StandardScaler()), ("m", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
    "SVM": Pipeline([( "scaler", StandardScaler()), ("m", SVC(kernel="rbf", C=5.0, gamma="scale", random_state=RANDOM_STATE))]),
    "Tree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
}

rows = []
preds = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    preds[name] = p
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, p),
        "macro_f1": f1_score(y_test, p, average="macro"),
    })

summary = pd.DataFrame(rows).sort_values(["macro_f1", "accuracy"], ascending=False)
display(summary.round(4))

In [ ]:
tree = models["Tree"]
imp = pd.Series(tree.feature_importances_, index=feature_names).sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=imp.head(10).values, y=imp.head(10).index, color="steelblue")
plt.title("DecisionTree: топ-10 важных признаков")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(tree, feature_names=feature_names, class_names=["0","1","2"], filled=True, rounded=True, fontsize=8)
plt.tight_layout()
plt.show()

print(export_text(tree, feature_names=feature_names)[:2500])